In [1]:
from pathlib import Path
from pypdf import PdfReader

PDF_DIR = Path("../data/pdfs")
pages = []

for pdf in PDF_DIR.glob("*.pdf"):
    reader = PdfReader(pdf)
    for i, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"doc": pdf.name, "page": i, "text": text})

print("Total pages with text:", len(pages))
print(pages[0]["doc"], pages[0]["page"], pages[0]["text"][:300])

Total pages with text: 30
CS201_Data_Structures_and_Algorithms.pdf 1 CS201
Page 1
CS201: Data Structures and Algorithms
Chapter 1: Algorithm Analysis and Big-O
An algorithm is a finite sequence of steps that solves a problem. To compare algorithms we
measure how their running time and memory use grow as the input size n grows. Big-O
notation describes the upper bound


In [2]:
def chunk_text(text, size=800, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = []
for p in pages:
    for j, c in enumerate(chunk_text(p["text"])):
        chunks.append({
            "id": f'{p["doc"]}-p{p["page"]}-c{j}',
            "text": c, "doc": p["doc"], "page": p["page"],
        })
print("Total chunks:", len(chunks))

Total chunks: 60


In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../chroma_db")
col = client.get_or_create_collection("course", metadata={"hnsw:space": "cosine"})

embeddings = model.encode([c["text"] for c in chunks], show_progress_bar=True).tolist()

col.upsert(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=embeddings,
    metadatas=[{"doc": c["doc"], "page": c["page"]} for c in chunks],
)
print("Stored:", col.count())

W1107 11:06:33.642000 7836 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


PydanticImportError: `BaseSettings` has been moved to the `pydantic-settings` package. See https://pydantic.dev/docs/validation/2.14/migration/#basesettings-has-moved-to-pydantic-settings for more details.

For further information visit https://errors.pydantic.dev/2.14/u/import-error

In [ ]:
def retrieve(question, k=4):
    q_emb = model.encode([question]).tolist()
    r = col.query(query_embeddings=q_emb, n_results=k)
    return [
        {"text": d, "doc": m["doc"], "page": m["page"], "score": round(1 - dist, 3)}
        for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])
    ]

# 1) course ka sawal
for r in retrieve("YAHAN apni course ka koi sawal likho"):
    print(r["score"], r["doc"], "p.", r["page"], "|", r["text"][:100].replace("\n", " "))

print("-----")
# 2) course se bahar ka sawal
for r in retrieve("Who won the FIFA World Cup in 2018?"):
    print(r["score"], r["doc"], "p.", r["page"], "|", r["text"][:100].replace("\n", " "))